# Phase 9 FINAL — Hybrid RAG Inference & Evaluation
## Qwen2.5 + Phase1/Phase2 merge + FAISS + BM25 + RRF + Cross-Encoder + Metadata + LangChain

Notebook này là bước inference/deployment sau Phase 8. Thiết kế ưu tiên **groundedness / chống bịa** thay vì chỉ tối đa khả năng trả lời.

Luồng chính:

```text
Question
  → OOD / scope guard
  → FAISS dense retrieval (E5)
  → BM25 lexical retrieval
  → Reciprocal Rank Fusion (RRF)
  → TOP 20 candidates
  → Cross-Encoder reranker
  → metadata soft boost (year/person/dynasty/event/...)
  → TOP context chunks
  → Qwen2.5: base → merge Phase1 → merge Phase2
  → source/citation + unsupported-year post-validation
  → final grounded answer / abstain
```

Các nguyên tắc quan trọng:

- Corpus chính: ưu tiên `vn_history_rag_chunks_enriched.jsonl` của Phase 8.
- `title + text` dùng cho FAISS/BM25/reranker; metadata chỉ **soft boost**, không lọc cứng.
- Qwen Phase2 vẫn nhận context theo format đã train: `[chunk_id] title\ntext`.
- Adapter được load/merge đúng thứ tự Phase 7: **vanilla → Phase1 merge → Phase2 merge**.
- Có guard trước và sau generation: câu lạc đề, source ID bịa, hoặc năm không có trong evidence đều có thể bị chặn.
- Benchmark 100 câu: tái tạo **held-out eval+test split Phase2**, lấy ngẫu nhiên 90 câu lịch sử + 10 câu OOD; so sánh 4 cấu hình.

> Benchmark này là diagnostic end-to-end. 90 câu lịch sử lấy từ held-out 10% của Phase2 theo đúng seed/split cũ, nên tốt hơn lấy ngẫu nhiên từ train set.

In [ ]:
# Cell 1 — Install dependencies
# Nếu Colab yêu cầu restart runtime sau pip, restart rồi chạy lại từ Cell 2.

%pip -q install -U \
  "transformers>=4.51,<5" \
  "peft>=0.13,<1" \
  "accelerate>=1.0,<2" \
  "sentence-transformers>=3.4,<6" \
  "faiss-cpu>=1.8" \
  "bm25s>=0.2.14" \
  "langchain-core>=0.3,<2" \
  "scikit-learn>=1.4" \
  "pandas>=2.0" \
  "safetensors>=0.4" \
  "tqdm>=4.66"

In [ ]:
# Cell 2 — Mount Drive, imports, global config

from google.colab import drive
drive.mount('/content/drive')

import os, re, gc, json, glob, time, math, shutil, zipfile, hashlib, random, unicodedata
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import faiss
import bm25s

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sklearn.model_selection import train_test_split
from langchain_core.runnables import RunnableLambda
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# =========================
# Main model / Drive paths
# =========================
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DRIVE_ROOT = Path('/content/drive/MyDrive')
BACKUP_DIR = DRIVE_ROOT / 'vn_history_model_backups'
RAG_ROOT = BACKUP_DIR / 'rag_corpus_vn_history'
PHASE8_METADATA_DIR = RAG_ROOT / 'metadata'
PHASE9_DIR = RAG_ROOT / 'phase9_hybrid_rag'
PHASE9_DIR.mkdir(parents=True, exist_ok=True)

ENRICHED_CANDIDATES = [
    PHASE8_METADATA_DIR / 'vn_history_rag_chunks_enriched.jsonl',
    RAG_ROOT / 'processed' / 'vn_history_rag_chunks_enriched.jsonl',
]
RAW_CORPUS_PATH = RAG_ROOT / 'processed' / 'vn_history_rag_chunks.jsonl'
METADATA_PATH = PHASE8_METADATA_DIR / 'vn_history_rag_chunk_metadata.jsonl'

# Phase2 diagnostic dataset — dùng để reconstruct eval/test benchmark.
SFT_DATA_DIR = DRIVE_ROOT / 'vn_history_rag_sft_dataset'
MESSAGES_PATH = SFT_DATA_DIR / 'all_messages.jsonl'

# Adapter candidates giống Phase 7.
PHASE1_ADAPTER_CANDIDATES = [
    BACKUP_DIR / 'qwen_vnhistory_phase1_best_adapter',
    BACKUP_DIR / 'qwen_vnhistory_phase1_best_adapter.zip',
]
PHASE2_ADAPTER_CANDIDATES = [
    BACKUP_DIR / 'qwen_vnhistory_phase2_rag_best_adapter',
    BACKUP_DIR / 'qwen_vnhistory_phase2_rag_best_adapter.zip',
    BACKUP_DIR / 'qwen2_5_3b_vnhistory_phase2_rag_qlora_best_by_generation_metric',
    BACKUP_DIR / 'qwen2_5_3b_vnhistory_phase2_rag_qlora_best_by_generation_metric.zip',
]

# =========================
# Retrieval configuration
# =========================
EMBEDDING_MODEL_ID = 'intfloat/multilingual-e5-base'
RERANKER_MODEL_ID = 'BAAI/bge-reranker-v2-m3'

EMBED_BATCH_SIZE = 96
DENSE_FETCH_K = 80
BM25_FETCH_K = 80
RRF_K = 60
RRF_TOP_K = 20              # yêu cầu: đúng 20 ứng viên sau fusion
FINAL_CONTEXT_K = 6
RERANK_BATCH_SIZE = 16

FORCE_REBUILD_FAISS = False
FORCE_REBUILD_BM25 = False

# Metadata chỉ boost mềm — không loại chunk.
METADATA_MAX_BONUS = 0.18

# =========================
# Generation / safety
# =========================
MAX_INPUT_TOKENS = 3600
MAX_NEW_TOKENS = 256
MAX_CHARS_PER_CHUNK = 1800
MIN_CHARS_PER_CHUNK = 550
TEMPERATURE = 0.0
TOP_P = 1.0
REPETITION_PENALTY = 1.05
STRICT_SOURCE_REQUIRED = True
STRICT_UNSUPPORTED_YEAR_GUARD = True

# OOD: ngưỡng này chỉ là một lớp guard; explicit OOD + anchor similarity mới block sớm.
OOD_ANCHOR_MARGIN = 0.02
SECONDARY_OOD_MARGIN = -0.06
SECONDARY_MIN_DENSE = 0.28

# Benchmark
BENCHMARK_HISTORY_N = 90
BENCHMARK_OOD_N = 10
BENCHMARK_MAX_NEW_TOKENS = 160
BENCHMARK_SAVE_EVERY = 5
BENCHMARK_PATH = PHASE9_DIR / 'benchmark_100.jsonl'
BENCHMARK_RESULTS_PATH = PHASE9_DIR / 'benchmark_results.jsonl'
BENCHMARK_SUMMARY_PATH = PHASE9_DIR / 'benchmark_summary.csv'

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('PHASE9_DIR:', PHASE9_DIR)

In [ ]:
# Cell 3 — Utilities: JSONL, normalization, corpus/adapters auto-resolve

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with Path(path).open('r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f'JSON lỗi {path} dòng {line_no}: {e}') from e
    return rows


def append_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
        f.flush()


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def clean_text(s: Any) -> str:
    s = '' if s is None else str(s)
    s = s.replace('\r\n', '\n').replace('\r', '\n')
    s = re.sub(r'[ \t]+', ' ', s)
    s = re.sub(r'\n{3,}', '\n\n', s)
    return s.strip()


def strip_accents(s: str) -> str:
    s = unicodedata.normalize('NFD', clean_text(s).lower())
    s = ''.join(ch for ch in s if unicodedata.category(ch) != 'Mn')
    return s.replace('đ', 'd')


def match_norm(s: str) -> str:
    s = strip_accents(s)
    s = re.sub(r'[^a-z0-9]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def short_text(s: str, max_chars: int) -> str:
    s = clean_text(s)
    if len(s) <= max_chars:
        return s
    cut = s[:max_chars]
    last = max(cut.rfind('. '), cut.rfind('; '), cut.rfind('\n'), cut.rfind(' '))
    if last > max_chars * 0.65:
        cut = cut[:last]
    return cut.strip() + ' ...'


def corpus_signature(rows: List[Dict[str, Any]]) -> str:
    h = hashlib.sha1()
    for r in rows:
        cid = str(r.get('chunk_id', ''))
        text_hash = str(r.get('text_hash', ''))
        if not text_hash:
            text_hash = hashlib.sha1((clean_text(r.get('title')) + '\n' + clean_text(r.get('text'))).encode('utf-8')).hexdigest()[:16]
        h.update(f'{cid}|{text_hash}\n'.encode('utf-8'))
    return h.hexdigest()[:20]


def resolve_enriched_corpus() -> Path:
    for p in ENRICHED_CANDIDATES:
        if p.exists() and p.stat().st_size > 0:
            return p

    # Fallback: join raw corpus + metadata bằng chunk_id nếu enriched chưa có.
    if RAW_CORPUS_PATH.exists() and METADATA_PATH.exists():
        print('Không thấy enriched file; sẽ join raw + metadata trong RAM.')
        return Path('__JOIN_RAW_METADATA__')

    raise FileNotFoundError(
        'Không tìm thấy enriched corpus, cũng không đủ raw+metadata để join.\n' +
        '\n'.join(map(str, ENRICHED_CANDIDATES + [RAW_CORPUS_PATH, METADATA_PATH]))
    )


def resolve_adapter_dir(name: str, candidates: List[Path]) -> str:
    """Giống Phase 7: nhận folder trực tiếp hoặc zip, tự extract nếu cần."""
    extract_root = Path('/content/adapters') / name
    extract_root.parent.mkdir(parents=True, exist_ok=True)

    for p in candidates:
        p = Path(p)
        if p.is_dir() and (p / 'adapter_config.json').exists():
            return str(p)
        if p.is_file() and p.suffix.lower() == '.zip':
            if (extract_root / 'adapter_config.json').exists():
                return str(extract_root)
            print(f'Extracting {name} adapter:', p)
            if extract_root.exists():
                shutil.rmtree(extract_root)
            extract_root.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(p, 'r') as z:
                z.extractall(extract_root)
            matches = list(extract_root.rglob('adapter_config.json'))
            if not matches:
                raise FileNotFoundError(f'Không thấy adapter_config.json trong {p}')
            return str(matches[0].parent)

    # fallback recursive search trong backup dir
    matches = list(BACKUP_DIR.rglob('adapter_config.json'))
    for m in matches:
        if name.lower() in str(m).lower():
            return str(m.parent)

    raise FileNotFoundError(f'Không tìm thấy adapter {name}. Candidates: {candidates}')


def get_model_device(model):
    try:
        return model.get_input_embeddings().weight.device
    except Exception:
        return next(model.parameters()).device


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

In [ ]:
# Cell 4 — Load enriched corpus (auto) + strict structural checks

resolved = resolve_enriched_corpus()

if str(resolved) == '__JOIN_RAW_METADATA__':
    raw_rows = read_jsonl(RAW_CORPUS_PATH)
    metadata_rows = read_jsonl(METADATA_PATH)
    metadata_by_id = {str(m['chunk_id']): m for m in metadata_rows}
    chunks = []
    for c in raw_rows:
        cid = str(c['chunk_id'])
        e = dict(c)
        e['metadata'] = metadata_by_id.get(cid, {})
        chunks.append(e)
    CORPUS_PATH = RAW_CORPUS_PATH
else:
    CORPUS_PATH = resolved
    chunks = read_jsonl(CORPUS_PATH)

if not chunks:
    raise RuntimeError('Corpus rỗng.')

required = {'chunk_id', 'title', 'text'}
for i, c in enumerate(chunks[:50]):
    miss = required - set(c)
    if miss:
        raise ValueError(f'Chunk {i} thiếu {sorted(miss)}')

chunk_ids = [str(c['chunk_id']) for c in chunks]
if len(set(chunk_ids)) != len(chunk_ids):
    raise ValueError('Corpus có chunk_id trùng.')

chunk_by_id = {str(c['chunk_id']): c for c in chunks}
CORPUS_ID_SET = set(chunk_by_id)
CORPUS_SIGNATURE = corpus_signature(chunks)

metadata_nonempty = sum(bool(c.get('metadata')) for c in chunks)
print('CORPUS_PATH:', CORPUS_PATH)
print('Chunks:', f'{len(chunks):,}')
print('Metadata present:', f'{metadata_nonempty:,}/{len(chunks):,}')
print('Corpus signature:', CORPUS_SIGNATURE)
print('Sample keys:', sorted(chunks[0].keys()))

In [ ]:
# Cell 5 — Build/load E5 FAISS index trên toàn bộ enriched corpus

safe_embed = re.sub(r'[^A-Za-z0-9._-]+', '_', EMBEDDING_MODEL_ID)
FAISS_DIR = PHASE9_DIR / f'faiss_{safe_embed}'
FAISS_DIR.mkdir(parents=True, exist_ok=True)
FAISS_INDEX_PATH = FAISS_DIR / 'chunks.index'
FAISS_MANIFEST_PATH = FAISS_DIR / 'manifest.json'

embed_device = 'cuda' if torch.cuda.is_available() else 'cpu'
embedder = SentenceTransformer(EMBEDDING_MODEL_ID, device=embed_device)
try:
    embedder.max_seq_length = 512
except Exception:
    pass


def passage_for_embedding(c: Dict[str, Any]) -> str:
    return 'passage: ' + clean_text(f"{c.get('title','')}\n{c.get('text','')}")


def query_for_embedding(q: str) -> str:
    return 'query: ' + clean_text(q)


def faiss_cache_valid() -> bool:
    if FORCE_REBUILD_FAISS or not FAISS_INDEX_PATH.exists() or not FAISS_MANIFEST_PATH.exists():
        return False
    try:
        manifest = json.loads(FAISS_MANIFEST_PATH.read_text(encoding='utf-8'))
        return (
            manifest.get('corpus_signature') == CORPUS_SIGNATURE
            and manifest.get('embedding_model') == EMBEDDING_MODEL_ID
            and int(manifest.get('count', -1)) == len(chunks)
        )
    except Exception:
        return False


if faiss_cache_valid():
    print('Loading FAISS cache:', FAISS_INDEX_PATH)
    faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))
else:
    print('Building FAISS embeddings for', f'{len(chunks):,}', 'chunks...')
    texts = [passage_for_embedding(c) for c in chunks]
    embeddings = embedder.encode(
        texts,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype('float32')

    faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
    faiss_index.add(embeddings)

    local_idx = Path('/content/phase9_chunks.index')
    faiss.write_index(faiss_index, str(local_idx))
    shutil.copy2(local_idx, FAISS_INDEX_PATH)

    write_json(FAISS_MANIFEST_PATH, {
        'corpus_signature': CORPUS_SIGNATURE,
        'embedding_model': EMBEDDING_MODEL_ID,
        'count': len(chunks),
        'dim': int(embeddings.shape[1]),
    })
    del embeddings, texts
    cleanup_cuda()

assert faiss_index.ntotal == len(chunks), (faiss_index.ntotal, len(chunks))
print('FAISS ready:', faiss_index.ntotal)

In [ ]:
# Cell 6 — Build/load BM25S index (memory-efficient BM25)

BM25_DIR = PHASE9_DIR / 'bm25s_index'
BM25_MANIFEST_PATH = BM25_DIR / 'phase9_manifest.json'


def bm25_text(c: Dict[str, Any]) -> str:
    # title lặp 2 lần để lexical retrieval ưu tiên entity/title nhưng vẫn giữ full text.
    title = clean_text(c.get('title', ''))
    text = clean_text(c.get('text', ''))
    return match_norm(f'{title} {title} {text}')


def bm25_cache_valid() -> bool:
    if FORCE_REBUILD_BM25 or not BM25_DIR.exists() or not BM25_MANIFEST_PATH.exists():
        return False
    try:
        manifest = json.loads(BM25_MANIFEST_PATH.read_text(encoding='utf-8'))
        return (
            manifest.get('corpus_signature') == CORPUS_SIGNATURE
            and int(manifest.get('count', -1)) == len(chunks)
        )
    except Exception:
        return False


if bm25_cache_valid():
    print('Loading BM25S mmap cache:', BM25_DIR)
    bm25_retriever = bm25s.BM25.load(str(BM25_DIR), mmap=True, load_corpus=False)
else:
    print('Building BM25S index...')
    BM25_DIR.mkdir(parents=True, exist_ok=True)
    lexical_corpus = [bm25_text(c) for c in tqdm(chunks, desc='Prepare BM25 text')]
    corpus_tokens = bm25s.tokenize(lexical_corpus, stopwords=None, stemmer=None)
    bm25_retriever = bm25s.BM25()
    bm25_retriever.index(corpus_tokens)
    bm25_retriever.save(str(BM25_DIR))
    write_json(BM25_MANIFEST_PATH, {
        'corpus_signature': CORPUS_SIGNATURE,
        'count': len(chunks),
        'note': 'title duplicated x2 + full text, normalized for Vietnamese lexical retrieval',
    })
    del lexical_corpus, corpus_tokens
    gc.collect()

print('BM25 ready.')

In [ ]:
# Cell 7 — Reranker + intent/OOD anchors + metadata helper

rerank_device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Loading reranker:', RERANKER_MODEL_ID)
reranker = CrossEncoder(RERANKER_MODEL_ID, max_length=512, device=rerank_device)

HISTORY_ANCHORS = [
    'lịch sử Việt Nam các triều đại và nhà nước',
    'khởi nghĩa kháng chiến chiến tranh trong lịch sử Việt Nam',
    'nhân vật lịch sử Việt Nam vua tướng lãnh tụ',
    'hiệp định ngoại giao cách mạng Việt Nam',
    'nhà Lý nhà Trần nhà Lê nhà Nguyễn',
    'Cách mạng tháng Tám chiến tranh Việt Nam',
]

OOD_ANCHORS = [
    'lập trình Python JavaScript sửa lỗi phần mềm',
    'thời tiết hôm nay nhiệt độ dự báo mưa',
    'bóng đá cầu thủ câu lạc bộ tỉ số trận đấu',
    'nấu ăn công thức món ăn nguyên liệu',
    'toán học phương trình tính toán',
    'triệu chứng bệnh thuốc điều trị y tế',
    'giá cổ phiếu tiền điện tử tài chính hôm nay',
    'tình yêu quan hệ cá nhân hẹn hò',
]

anchor_texts = ['query: ' + x for x in HISTORY_ANCHORS + OOD_ANCHORS]
anchor_embs = embedder.encode(anchor_texts, convert_to_numpy=True, normalize_embeddings=True).astype('float32')
HISTORY_ANCHOR_EMBS = anchor_embs[:len(HISTORY_ANCHORS)]
OOD_ANCHOR_EMBS = anchor_embs[len(HISTORY_ANCHORS):]

# Patterns chạy trên match_norm(question), nên viết KHÔNG DẤU.
EXPLICIT_OOD_PATTERNS = [
    r'\bpython\b|\bjavascript\b|\bjava\b|lap trinh|viet code|debug|\bapi\b',
    r'thoi tiet|nhiet do hom nay|du bao mua|do am hom nay',
    r'\bmessi\b|\bronaldo\b|premier league|champions league|ti so bong da|world cup 20\d\d',
    r'cong thuc nau|nau mon|chien bao lau|luoc bao lau',
    r'giai phuong trinh|dao ham|tich phan|tinh \d+\s*[+*/-]',
    r'trieu chung|lieu thuoc|uong thuoc|dau bung|sot bao nhieu',
    r'gia bitcoin|gia co phieu|ty gia hom nay|mua coin',
    r'iphone|samsung|dien thoai nao',
    r'dich cau|dich sang tieng viet|translate',
]


def intent_scores(question: str) -> Dict[str, Any]:
    q_emb = embedder.encode([query_for_embedding(question)], convert_to_numpy=True, normalize_embeddings=True).astype('float32')[0]
    hs = float(np.max(HISTORY_ANCHOR_EMBS @ q_emb))
    oscore = float(np.max(OOD_ANCHOR_EMBS @ q_emb))
    qn = match_norm(question)
    explicit = any(re.search(p, qn, flags=re.IGNORECASE) for p in EXPLICIT_OOD_PATTERNS)
    return {
        'history_anchor': hs,
        'ood_anchor': oscore,
        'margin': hs - oscore,
        'explicit_ood': bool(explicit),
        'query_embedding': q_emb,
    }


def metadata_bonus(question: str, c: Dict[str, Any]) -> Tuple[float, List[str]]:
    md = c.get('metadata') or {}
    qn = match_norm(question)
    qyears = {int(x) for x in re.findall(r'(?<!\d)(\d{3,4})(?!\d)', question)}

    bonus = 0.0
    hits = []

    # years: mạnh nhưng vẫn soft.
    years = {int(y) for y in md.get('years', []) if str(y).isdigit()}
    year_hits = sorted(qyears & years)
    if year_hits:
        bonus += min(0.07, 0.035 * len(year_hits))
        hits.append('years=' + ','.join(map(str, year_hits)))

    field_weights = {
        'people': 0.055,
        'events': 0.045,
        'documents': 0.055,
        'dynasties': 0.055,
        'locations': 0.030,
        'periods': 0.030,
        'topics': 0.012,
        'content_facets': 0.010,
    }

    for field, weight in field_weights.items():
        matched = []
        for item in md.get(field, []) or []:
            ni = match_norm(item)
            if len(ni) >= 4 and re.search(rf'(?<![a-z0-9]){re.escape(ni)}(?![a-z0-9])', qn):
                matched.append(str(item))
        if matched:
            bonus += weight * min(2, len(matched))
            hits.append(f"{field}=" + '|'.join(matched[:2]))

    return min(METADATA_MAX_BONUS, bonus), hits

In [ ]:
# Cell 8 — Hybrid retrieval: FAISS + BM25 → RRF → 20 → reranker → metadata soft boost


def dense_search(question: str, k: int = DENSE_FETCH_K) -> List[Tuple[int, float]]:
    q_emb = embedder.encode(
        [query_for_embedding(question)],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype('float32')
    scores, idxs = faiss_index.search(q_emb, min(k, faiss_index.ntotal))
    return [(int(i), float(s)) for i, s in zip(idxs[0], scores[0]) if int(i) >= 0]


def bm25_search(question: str, k: int = BM25_FETCH_K) -> List[Tuple[int, float]]:
    qt = bm25s.tokenize([match_norm(question)], stopwords=None, stemmer=None)
    idxs, scores = bm25_retriever.retrieve(qt, k=min(k, len(chunks)))
    return [(int(i), float(s)) for i, s in zip(np.asarray(idxs[0]).tolist(), np.asarray(scores[0]).tolist())]


def rrf_fuse(dense: List[Tuple[int, float]], lexical: List[Tuple[int, float]], top_k: int = RRF_TOP_K) -> List[Dict[str, Any]]:
    fused = defaultdict(float)
    info = defaultdict(dict)

    for rank, (idx, score) in enumerate(dense, 1):
        fused[idx] += 1.0 / (RRF_K + rank)
        info[idx]['dense_rank'] = rank
        info[idx]['dense_score'] = score

    for rank, (idx, score) in enumerate(lexical, 1):
        fused[idx] += 1.0 / (RRF_K + rank)
        info[idx]['bm25_rank'] = rank
        info[idx]['bm25_score'] = score

    ordered = sorted(fused, key=fused.get, reverse=True)[:top_k]
    out = []
    for idx in ordered:
        c = dict(chunks[idx])
        c['_corpus_idx'] = idx
        c['rrf_score'] = float(fused[idx])
        c.update(info[idx])
        out.append(c)
    return out


def minmax(values: List[float]) -> np.ndarray:
    arr = np.asarray(values, dtype=np.float32)
    if len(arr) == 0:
        return arr
    lo, hi = float(arr.min()), float(arr.max())
    if hi - lo < 1e-9:
        return np.ones_like(arr) * 0.5
    return (arr - lo) / (hi - lo)


def hybrid_retrieve(question: str, final_k: int = FINAL_CONTEXT_K) -> Dict[str, Any]:
    question = clean_text(question)
    intent = intent_scores(question)

    # Chỉ block sớm khi vừa có explicit OOD cue vừa nghiêng về OOD anchors.
    if intent['explicit_ood'] and intent['margin'] < OOD_ANCHOR_MARGIN:
        return {
            'question': question,
            'is_ood': True,
            'ood_reason': 'explicit_ood+anchor_guard',
            'intent': {k:v for k,v in intent.items() if k != 'query_embedding'},
            'candidates20': [],
            'final_context': [],
            'max_dense': None,
        }

    dense = dense_search(question)
    lexical = bm25_search(question)
    max_dense = dense[0][1] if dense else -1.0

    # Secondary OOD guard: retrieval yếu + anchors nghiêng rõ sang OOD.
    if intent['margin'] < SECONDARY_OOD_MARGIN and max_dense < SECONDARY_MIN_DENSE:
        return {
            'question': question,
            'is_ood': True,
            'ood_reason': 'weak_history_retrieval+anchor_guard',
            'intent': {k:v for k,v in intent.items() if k != 'query_embedding'},
            'candidates20': [],
            'final_context': [],
            'max_dense': float(max_dense),
        }

    candidates = rrf_fuse(dense, lexical, top_k=RRF_TOP_K)
    if not candidates:
        return {
            'question': question,
            'is_ood': False,
            'ood_reason': '',
            'intent': {k:v for k,v in intent.items() if k != 'query_embedding'},
            'candidates20': [],
            'final_context': [],
            'max_dense': float(max_dense),
        }

    pairs = [
        [question, clean_text(f"{c.get('title','')}\n{c.get('text','')}")]
        for c in candidates
    ]
    rr_scores = reranker.predict(
        pairs,
        batch_size=RERANK_BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    rr_scores = np.asarray(rr_scores).reshape(-1).astype(float)
    rr_norm = minmax(rr_scores.tolist())
    rrf_norm = minmax([c['rrf_score'] for c in candidates])

    for i, c in enumerate(candidates):
        bonus, hits = metadata_bonus(question, c)
        c['reranker_score'] = float(rr_scores[i])
        c['reranker_norm'] = float(rr_norm[i])
        c['rrf_norm'] = float(rrf_norm[i])
        c['metadata_bonus'] = float(bonus)
        c['metadata_hits'] = hits
        # metadata chỉ là boost nhỏ sau semantic reranker.
        c['final_retrieval_score'] = float(0.72 * rr_norm[i] + 0.28 * rrf_norm[i] + bonus)

    candidates.sort(key=lambda x: x['final_retrieval_score'], reverse=True)
    final_context = candidates[:final_k]

    return {
        'question': question,
        'is_ood': False,
        'ood_reason': '',
        'intent': {k:v for k,v in intent.items() if k != 'query_embedding'},
        'candidates20': candidates,
        'final_context': final_context,
        'max_dense': float(max_dense),
    }


# Retrieval smoke test — chưa cần Qwen.
_smoke = hybrid_retrieve('Bình Ngô đại cáo ra đời trong bối cảnh nào?')
print('OOD:', _smoke['is_ood'])
for i, c in enumerate(_smoke['final_context'], 1):
    print(i, c['chunk_id'], '|', c.get('title'), '| score=', round(c['final_retrieval_score'], 4), '| md=', c.get('metadata_hits'))

In [ ]:
# Cell 9 — Qwen model loaders: vanilla → Phase1 merge → Phase2 merge (đúng Phase 7)

PHASE1_ADAPTER_DIR = resolve_adapter_dir('phase1', PHASE1_ADAPTER_CANDIDATES)
PHASE2_ADAPTER_DIR = resolve_adapter_dir('phase2', PHASE2_ADAPTER_CANDIDATES)
print('PHASE1_ADAPTER_DIR:', PHASE1_ADAPTER_DIR)
print('PHASE2_ADAPTER_DIR:', PHASE2_ADAPTER_DIR)

IM_START = '<|im_start|>'
IM_END = '<|im_end|>'

DEFAULT_SYSTEM = (
    'Bạn là trợ lý AI chuyên về lịch sử Việt Nam. '
    'Hãy suy luận cẩn thận, trả lời chính xác, rõ ràng và không bịa thông tin.'
)


def load_tokenizer():
    tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'
    return tok


def load_base_model():
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map='auto' if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )


def load_model_variant(variant: str):
    """
    variant:
      vanilla   = base Qwen2.5
      stage1    = base + Phase1 merge
      stage12   = base + Phase1 merge + Phase2 merge
    """
    variant = variant.lower()
    if variant not in {'vanilla', 'stage1', 'stage12'}:
        raise ValueError(variant)

    tok = load_tokenizer()
    model = load_base_model()
    model.config.pad_token_id = tok.pad_token_id

    if variant in {'stage1', 'stage12'}:
        print('Loading Phase1 adapter → merge...')
        model = PeftModel.from_pretrained(model, PHASE1_ADAPTER_DIR, is_trainable=False)
        model = model.merge_and_unload()
        cleanup_cuda()

    if variant == 'stage12':
        print('Loading Phase2 adapter on Phase1-merged model → merge...')
        model = PeftModel.from_pretrained(model, PHASE2_ADAPTER_DIR, is_trainable=False)
        model = model.merge_and_unload()
        cleanup_cuda()

    model.eval()
    model.config.use_cache = True
    model.config.pad_token_id = tok.pad_token_id
    return model, tok


@torch.inference_mode()
def generate_raw(model, tok, prompt: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    device = get_model_device(model)
    inputs = tok(prompt, return_tensors='pt', add_special_tokens=False).to(device)
    im_end_id = tok.convert_tokens_to_ids(IM_END)
    eos_ids = [tok.eos_token_id]
    if isinstance(im_end_id, int) and im_end_id >= 0:
        eos_ids.append(im_end_id)

    kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=TEMPERATURE > 0,
        top_p=TOP_P,
        repetition_penalty=REPETITION_PENALTY,
        eos_token_id=eos_ids,
        pad_token_id=tok.pad_token_id,
    )
    if TEMPERATURE > 0:
        kwargs['temperature'] = TEMPERATURE

    out = model.generate(**kwargs)
    gen = out[0][inputs['input_ids'].shape[-1]:]
    return tok.decode(gen, skip_special_tokens=False)


def clean_generated(text: str, tok) -> str:
    text = clean_text(text)
    for marker in [IM_END, f'{IM_START}user', f'{IM_START}system', f'{IM_START}assistant']:
        if marker and marker in text:
            text = text.split(marker, 1)[0]
    for special in [IM_START, IM_END, tok.eos_token or '', tok.pad_token or '']:
        if special:
            text = text.replace(special, '')
    return clean_text(text)


def build_plain_prompt(question: str) -> str:
    return (
        f'{IM_START}system\n{DEFAULT_SYSTEM}{IM_END}\n'
        f'{IM_START}user\n{clean_text(question)}{IM_END}\n'
        f'{IM_START}assistant\n'
    )


def extract_plain_answer(raw: str, tok) -> str:
    raw = clean_generated(raw, tok)
    if '<final>' in raw:
        part = raw.split('<final>', 1)[1]
        if '</final>' in part:
            part = part.split('</final>', 1)[0]
        return clean_text(part)
    return raw


def build_context_text(contexts: List[Dict[str, Any]], chars_per_chunk: int) -> str:
    parts = []
    for c in contexts:
        parts.append(f"[{c['chunk_id']}] {clean_text(c.get('title',''))}\n{short_text(c.get('text',''), chars_per_chunk)}")
    return '\n\n'.join(parts)


def build_rag_user_text(question: str, contexts: List[Dict[str, Any]], chars_per_chunk: int) -> str:
    return (
        f'Câu hỏi:\n{clean_text(question)}\n\n'
        f'Tài liệu tham khảo:\n{build_context_text(contexts, chars_per_chunk)}'
    ).strip()


def build_rag_prompt(user_text: str) -> str:
    # Phase2/Phase7 style: KHÔNG thêm system.
    return f'{IM_START}user\n{user_text}{IM_END}\n{IM_START}assistant\n'


def fit_rag_prompt(tok, question: str, contexts: List[Dict[str, Any]]) -> Tuple[str, List[Dict[str, Any]], Dict[str, int]]:
    used = list(contexts)
    chars = MAX_CHARS_PER_CHUNK
    while used:
        prompt = build_rag_prompt(build_rag_user_text(question, used, chars))
        n = len(tok(prompt, add_special_tokens=False)['input_ids'])
        if n <= MAX_INPUT_TOKENS:
            return prompt, used, {'input_tokens': n, 'chars_per_chunk': chars, 'n_context': len(used)}
        if chars > MIN_CHARS_PER_CHUNK:
            chars = max(MIN_CHARS_PER_CHUNK, int(chars * 0.75))
        else:
            used = used[:-1]

    prompt = build_rag_prompt(build_rag_user_text(question, [], 0))
    return prompt, [], {'input_tokens': len(tok(prompt, add_special_tokens=False)['input_ids']), 'chars_per_chunk': 0, 'n_context': 0}


SOURCE_BLOCK_RE = re.compile(r'Nguồn được dùng\s*:\s*(.*?)(?:\n\s*\n|\n\s*Trả lời\s*:|$)', re.I | re.S)
ANSWER_SPLIT_RE = re.compile(r'Trả lời\s*:', re.I)


def parse_rag_output(raw: str, tok) -> Dict[str, Any]:
    cleaned = clean_generated(raw, tok)
    source_ids = []
    m = SOURCE_BLOCK_RE.search(cleaned)
    if m:
        block = m.group(1)
        groups = re.findall(r'\[([^\[\]]*)\]', block)
        if not groups and block.strip():
            groups = [block]
        for g in groups:
            for x in re.split(r'[,;\n]+', g):
                x = x.strip().strip('[]"\' ')
                if x:
                    source_ids.append(x)
    source_ids = list(dict.fromkeys(source_ids))

    parts = ANSWER_SPLIT_RE.split(cleaned, maxsplit=1)
    answer = clean_text(parts[1] if len(parts) > 1 else cleaned)
    return {
        'raw_output': cleaned,
        'source_ids': source_ids,
        'answer': answer,
        'format_ok': bool(re.search(r'Nguồn được dùng\s*:', cleaned, re.I) and re.search(r'Trả lời\s*:', cleaned, re.I)),
    }


REFUSAL_PATTERNS = [
    r'không đủ (?:thông tin|bằng chứng|dữ liệu)',
    r'tài liệu (?:được cung cấp )?(?:không|chưa) (?:nêu|cho biết|cung cấp)',
    r'không thể trả lời (?:chắc chắn|chính xác)?',
    r'ngoài phạm vi',
    r'không liên quan đến lịch sử việt nam',
]


def is_refusal(text: str) -> bool:
    return any(re.search(p, clean_text(text), flags=re.I) for p in REFUSAL_PATTERNS)

In [ ]:
# Cell 10 — Load full Phase1+Phase2 model + LangChain orchestration + anti-hallucination post-guard

print('Loading FINAL Stage1+Stage2 merged model...')
generation_model, generation_tokenizer = load_model_variant('stage12')
print('Final model device:', get_model_device(generation_model))

SAFE_OOD_ANSWER = 'Câu hỏi này nằm ngoài phạm vi hệ thống lịch sử Việt Nam, nên tôi không trả lời bằng corpus hiện tại.'
SAFE_INSUFFICIENT_ANSWER = 'Không đủ bằng chứng trong các tài liệu truy xuất để trả lời chắc chắn câu hỏi này.'


def extract_year_set(text: str) -> set:
    return {int(x) for x in re.findall(r'(?<!\d)(\d{3,4})(?!\d)', clean_text(text))}


def answer_support_score(answer: str, source_chunks: List[Dict[str, Any]]) -> Optional[float]:
    if not answer or not source_chunks:
        return None
    a = embedder.encode(['query: ' + answer], convert_to_numpy=True, normalize_embeddings=True).astype('float32')[0]
    passages = [passage_for_embedding(c) for c in source_chunks]
    p = embedder.encode(passages, convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    return float(np.max(p @ a))


def full_rag_answer_from_state(state: Dict[str, Any]) -> Dict[str, Any]:
    question = state['question']
    retrieval = state['retrieval']
    started = time.perf_counter()

    if retrieval.get('is_ood'):
        return {
            'question': question,
            'answer': SAFE_OOD_ANSWER,
            'status': 'blocked_off_topic',
            'source_ids': [],
            'invalid_source_ids': [],
            'unsupported_years': [],
            'format_ok': True,
            'retrieval': retrieval,
            'support_score': None,
            'latency_sec': time.perf_counter() - started,
        }

    contexts = retrieval.get('final_context', [])
    if not contexts:
        return {
            'question': question,
            'answer': SAFE_INSUFFICIENT_ANSWER,
            'status': 'blocked_no_context',
            'source_ids': [],
            'invalid_source_ids': [],
            'unsupported_years': [],
            'format_ok': True,
            'retrieval': retrieval,
            'support_score': None,
            'latency_sec': time.perf_counter() - started,
        }

    prompt, used_context, budget = fit_rag_prompt(generation_tokenizer, question, contexts)
    raw = generate_raw(generation_model, generation_tokenizer, prompt, max_new_tokens=MAX_NEW_TOKENS)
    parsed = parse_rag_output(raw, generation_tokenizer)

    allowed_ids = {str(c['chunk_id']) for c in used_context}
    valid_ids = [cid for cid in parsed['source_ids'] if cid in allowed_ids]
    invalid_ids = [cid for cid in parsed['source_ids'] if cid not in allowed_ids]

    evidence_chunks = [chunk_by_id[cid] for cid in valid_ids if cid in chunk_by_id]
    evidence_text = '\n'.join(clean_text(c.get('title','')) + '\n' + clean_text(c.get('text','')) for c in evidence_chunks)
    answer_years = extract_year_set(parsed['answer'])
    evidence_years = extract_year_set(evidence_text)
    unsupported_years = sorted(answer_years - evidence_years) if evidence_chunks else sorted(answer_years)

    status = 'ok'
    answer = parsed['answer']

    if invalid_ids:
        status = 'blocked_invalid_source_id'
        answer = SAFE_INSUFFICIENT_ANSWER
        valid_ids = []
    elif STRICT_SOURCE_REQUIRED and not valid_ids and not is_refusal(answer):
        status = 'blocked_missing_source'
        answer = SAFE_INSUFFICIENT_ANSWER
    elif STRICT_UNSUPPORTED_YEAR_GUARD and unsupported_years and not is_refusal(answer):
        status = 'blocked_unsupported_year'
        answer = SAFE_INSUFFICIENT_ANSWER
        valid_ids = []

    support = answer_support_score(parsed['answer'], evidence_chunks) if evidence_chunks else None

    return {
        'question': question,
        'answer': answer,
        'status': status,
        'source_ids': valid_ids,
        'model_source_ids': parsed['source_ids'],
        'invalid_source_ids': invalid_ids,
        'unsupported_years': unsupported_years,
        'format_ok': parsed['format_ok'],
        'raw_output': parsed['raw_output'],
        'retrieval': retrieval,
        'prompt_budget': budget,
        'support_score': support,
        'latency_sec': time.perf_counter() - started,
    }


# LangChain: normalize → hybrid retrieval → guarded generation/post-validation
rag_chain = (
    RunnableLambda(lambda q: {'question': clean_text(q)})
    | RunnableLambda(lambda s: {**s, 'retrieval': hybrid_retrieve(s['question'])})
    | RunnableLambda(full_rag_answer_from_state)
)


def ask_history(question: str, verbose: bool = True) -> Dict[str, Any]:
    result = rag_chain.invoke(question)
    if verbose:
        print('\nQUESTION:', result['question'])
        print('STATUS:', result['status'])
        print('ANSWER:', result['answer'])
        print('SOURCES:', result.get('source_ids', []))
        if result.get('unsupported_years'):
            print('UNSUPPORTED YEARS BLOCKED:', result['unsupported_years'])
        if result.get('invalid_source_ids'):
            print('INVALID SOURCE IDS BLOCKED:', result['invalid_source_ids'])
        retr = result.get('retrieval', {})
        if retr.get('final_context'):
            print('\nTop contexts:')
            for i, c in enumerate(retr['final_context'], 1):
                print(f" {i}. {c['chunk_id']} | {c.get('title','')} | final={c.get('final_retrieval_score',0):.4f} | md={c.get('metadata_hits',[])}")
    return result

print('LangChain RAG ready.')

In [ ]:
# Cell 11 — Safety smoke test: 1 lịch sử + 1 lạc đề

_ = ask_history('Chiến thắng Bạch Đằng năm 938 có ý nghĩa lịch sử gì?', verbose=True)
print('\n' + '='*100)
_ = ask_history('Viết cho tôi một hàm Python sắp xếp danh sách số nguyên.', verbose=True)

In [ ]:
# Cell 12 — Manual test 10 câu
# User đưa 9 câu; câu thứ 10 được bổ sung để đủ đúng 10 samples.

test_questions = [
    'Chiến thắng Bạch Đằng năm 938 có ý nghĩa lịch sử gì?',
    'Việc Lý Công Uẩn dời đô ra Thăng Long năm 1010 có ý nghĩa gì?',
    'Bình Ngô đại cáo ra đời trong bối cảnh nào?',
    'Khởi nghĩa Lam Sơn diễn ra trong bối cảnh nào và kết quả ra sao?',
    'So sánh vai trò của nhà Lý và nhà Trần trong xây dựng và bảo vệ Đại Việt.',
    'Phong trào Cần Vương bùng nổ trong hoàn cảnh nào?',
    'Xô Viết Nghệ Tĩnh 1930-1931 có đặc điểm gì nổi bật?',
    'Cách mạng tháng Tám năm 1945 diễn ra như thế nào và kết quả ra sao?',
    'Hiệp định Genève năm 1954 có nội dung/kết quả gì đối với Việt Nam?',
    'Ba lần kháng chiến chống Mông - Nguyên dưới nhà Trần có đặc điểm và ý nghĩa gì?',
]

manual_results = []
for i, q in enumerate(test_questions, 1):
    print('\n' + '='*120)
    print(f'MANUAL {i}/10')
    manual_results.append(ask_history(q, verbose=True))

manual_df = pd.DataFrame([{
    'question': r['question'],
    'status': r['status'],
    'answer': r['answer'],
    'sources': ','.join(r.get('source_ids', [])),
    'support_score': r.get('support_score'),
    'latency_sec': r.get('latency_sec'),
} for r in manual_results])
display(manual_df)

In [ ]:
# Cell 13 — Interactive: tự nhập một câu hỏi

question = input('Nhập câu hỏi lịch sử Việt Nam: ').strip()
if question:
    interactive_result = ask_history(question, verbose=True)
else:
    print('Bạn chưa nhập câu hỏi.')

In [ ]:
# Cell 14 — Build benchmark 100: reconstruct Phase2 held-out 10% → random 90 + 10 OOD

if not MESSAGES_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy {MESSAGES_PATH}')

messages = read_jsonl(MESSAGES_PATH)

QUESTION_RE = re.compile(r'Câu hỏi:\s*(.*?)(?:\n\s*\n\s*Tài liệu tham khảo:|$)', re.I | re.S)
SOURCE_LINE_RE = re.compile(r'Nguồn được dùng:\s*\[(.*?)\]', re.I | re.S)


def parse_message_sample(sample: Dict[str, Any], idx: int) -> Optional[Dict[str, Any]]:
    user_text, assistant_text = '', ''
    for m in sample.get('messages', []):
        if m.get('role') == 'user':
            user_text = m.get('content', '')
        elif m.get('role') == 'assistant':
            assistant_text = m.get('content', '')
    if not user_text or not assistant_text:
        return None

    qm = QUESTION_RE.search(user_text)
    question = clean_text(qm.group(1) if qm else user_text)

    sm = SOURCE_LINE_RE.search(assistant_text)
    gold_ids = []
    if sm:
        inside = sm.group(1).strip()
        if inside:
            gold_ids = [x.strip().strip('"\' ') for x in re.split(r'[,;]+', inside) if x.strip()]

    ans_parts = ANSWER_SPLIT_RE.split(assistant_text, maxsplit=1)
    answer_body = clean_text(ans_parts[1] if len(ans_parts) > 1 else assistant_text)

    return {
        'id': sample.get('id', f'sample_{idx:04d}'),
        'type': sample.get('type', 'unknown'),
        'question': question,
        'reference_answer': answer_body,
        'gold_source_ids': gold_ids,
    }

records = [r for i, s in enumerate(messages) if (r := parse_message_sample(s, i)) is not None]
df = pd.DataFrame(records)

# Tái tạo chính xác split Phase2: 90/5/5, stratified, seed 42.
train_df, temp_df = train_test_split(
    df,
    test_size=0.10,
    random_state=SEED,
    stratify=df['type'],
)
eval_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df['type'],
)
heldout = pd.concat([eval_df, test_df], ignore_index=True)

print('Reconstructed train/eval/test:', len(train_df), len(eval_df), len(test_df))
print('Held-out type counts:')
print(heldout['type'].value_counts())

history_n = min(BENCHMARK_HISTORY_N, len(heldout))
history_bench = heldout.sample(n=history_n, random_state=SEED).copy()
history_bench['is_off_topic'] = False
history_bench['expected_refusal'] = history_bench['type'].isin(['insufficient_context', 'false_premise'])

OOD_QUESTIONS = [
    'Viết cho tôi một hàm Python để merge hai dictionary.',
    'Thời tiết Thành phố Hồ Chí Minh hôm nay có mưa không?',
    'Messi đã ghi bao nhiêu bàn ở mùa giải gần nhất?',
    'Cách nấu bò kho ngon tại nhà như thế nào?',
    'Giải phương trình x^2 - 5x + 6 = 0.',
    'Đau đầu và sốt nhẹ thì nên uống thuốc gì?',
    'Giá Bitcoin hôm nay là bao nhiêu?',
    'Tôi nên làm gì khi người yêu không trả lời tin nhắn?',
    'Dịch câu "machine learning is useful" sang tiếng Việt.',
    'So sánh iPhone và Samsung đời mới nhất.',
][:BENCHMARK_OOD_N]

ood_rows = []
for i, q in enumerate(OOD_QUESTIONS, 1):
    ood_rows.append({
        'id': f'ood_{i:02d}',
        'type': 'off_topic',
        'question': q,
        'reference_answer': '',
        'gold_source_ids': [],
        'is_off_topic': True,
        'expected_refusal': True,
    })

benchmark_df = pd.concat([history_bench, pd.DataFrame(ood_rows)], ignore_index=True)
benchmark_df = benchmark_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Mark gold IDs còn tồn tại trong corpus Phase8 hiện tại.
benchmark_df['gold_source_ids_present'] = benchmark_df['gold_source_ids'].apply(
    lambda xs: [x for x in (xs or []) if x in CORPUS_ID_SET]
)
benchmark_df['missing_gold_ids'] = benchmark_df.apply(
    lambda r: [x for x in (r['gold_source_ids'] or []) if x not in CORPUS_ID_SET], axis=1
)

with BENCHMARK_PATH.open('w', encoding='utf-8') as f:
    for row in benchmark_df.to_dict('records'):
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('Benchmark size:', len(benchmark_df))
print('Off-topic:', int(benchmark_df['is_off_topic'].sum()))
print('Expected refusal:', int(benchmark_df['expected_refusal'].sum()))
print('Rows with missing old gold IDs:', int(benchmark_df['missing_gold_ids'].apply(bool).sum()))
print('Saved:', BENCHMARK_PATH)
display(benchmark_df[['id','type','question','expected_refusal']].head(12))

In [ ]:
# Cell 15 — Evaluation metrics

def metric_tokens(text: str) -> List[str]:
    return re.findall(r"[0-9A-Za-zÀ-ỹĐđ]+", clean_text(text).lower())


def token_f1(pred: str, ref: str) -> float:
    p, r = metric_tokens(pred), metric_tokens(ref)
    if not p and not r:
        return 1.0
    if not p or not r:
        return 0.0
    cp, cr = Counter(p), Counter(r)
    common = sum((cp & cr).values())
    precision = common / len(p)
    recall = common / len(r)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


def rouge_l_f1(pred: str, ref: str) -> float:
    a, b = metric_tokens(pred), metric_tokens(ref)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    # LCS with rolling DP.
    if len(b) > len(a):
        a, b = b, a
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b, 1):
            cur.append(prev[j-1] + 1 if x == y else max(prev[j], cur[-1]))
        prev = cur
    lcs = prev[-1]
    p = lcs / len(a)
    r = lcs / len(b)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def year_f1(pred: str, ref: str) -> float:
    p, r = extract_year_set(pred), extract_year_set(ref)
    if not p and not r:
        return 1.0
    if not p or not r:
        return 0.0
    common = len(p & r)
    pr = common / len(p)
    rc = common / len(r)
    return 0.0 if pr + rc == 0 else 2 * pr * rc / (pr + rc)


def source_prf(pred_ids: List[str], gold_ids: List[str]) -> Tuple[float,float,float]:
    p, g = set(pred_ids or []), set(gold_ids or [])
    if not p and not g:
        return 1.0, 1.0, 1.0
    precision = len(p & g) / len(p) if p else 0.0
    recall = len(p & g) / len(g) if g else (1.0 if not p else 0.0)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return precision, recall, f1


def retrieval_metrics(candidate20: List[str], final_ids: List[str], gold_ids: List[str]) -> Dict[str, float]:
    gold = [x for x in (gold_ids or []) if x in CORPUS_ID_SET]
    if not gold:
        return {'retrieval_recall20': np.nan, 'retrieval_recall_final': np.nan, 'mrr20': np.nan}
    gs = set(gold)
    recall20 = len(gs & set(candidate20)) / len(gs)
    recall_final = len(gs & set(final_ids)) / len(gs)
    rr = 0.0
    for rank, cid in enumerate(candidate20, 1):
        if cid in gs:
            rr = 1.0 / rank
            break
    return {'retrieval_recall20': recall20, 'retrieval_recall_final': recall_final, 'mrr20': rr}


def semantic_similarity_batch(preds: List[str], refs: List[str]) -> List[float]:
    if not preds:
        return []
    pe = embedder.encode(['query: ' + clean_text(x) for x in preds], batch_size=64, convert_to_numpy=True, normalize_embeddings=True)
    re_ = embedder.encode(['passage: ' + clean_text(x) for x in refs], batch_size=64, convert_to_numpy=True, normalize_embeddings=True)
    return np.sum(pe * re_, axis=1).astype(float).tolist()

In [ ]:
# Cell 16 — Benchmark runner: 4 configurations, checkpoint/resume
# 1) vanilla base, direct QA
# 2) Stage1 merged, direct QA
# 3) Stage1+Stage2 merged weights-only, EMPTY RAG context
# 4) Stage1+Stage2 merged + full Hybrid RAG / guards
#
# CẢNH BÁO: 4 x 100 generations có thể mất khá lâu. Kết quả được append checkpoint sau mỗi vài mẫu.

benchmark_rows = benchmark_df.to_dict('records')


def load_benchmark_done() -> Dict[Tuple[str,str], Dict[str,Any]]:
    if not BENCHMARK_RESULTS_PATH.exists():
        return {}
    out = {}
    for r in read_jsonl(BENCHMARK_RESULTS_PATH):
        out[(r['variant'], r['id'])] = r
    return out


def run_plain_variant(model, tok, row: Dict[str,Any], variant: str) -> Dict[str,Any]:
    started = time.perf_counter()
    prompt = build_plain_prompt(row['question'])
    raw = generate_raw(model, tok, prompt, max_new_tokens=BENCHMARK_MAX_NEW_TOKENS)
    answer = extract_plain_answer(raw, tok)
    return {
        'answer': answer,
        'raw_output': clean_generated(raw, tok),
        'source_ids': [],
        'format_ok': None,
        'status': 'plain_generation',
        'invalid_source_ids': [],
        'unsupported_years': [],
        'support_score': None,
        'candidate20_ids': [],
        'final_context_ids': [],
        'latency_sec': time.perf_counter() - started,
    }


def run_stage12_weights_only(model, tok, row: Dict[str,Any]) -> Dict[str,Any]:
    started = time.perf_counter()
    prompt = build_rag_prompt(build_rag_user_text(row['question'], [], 0))
    raw = generate_raw(model, tok, prompt, max_new_tokens=BENCHMARK_MAX_NEW_TOKENS)
    parsed = parse_rag_output(raw, tok)
    return {
        'answer': parsed['answer'],
        'raw_output': parsed['raw_output'],
        'source_ids': parsed['source_ids'],
        'format_ok': parsed['format_ok'],
        'status': 'stage12_no_context',
        'invalid_source_ids': parsed['source_ids'],  # không có context nên mọi citation đều invalid
        'unsupported_years': list(extract_year_set(parsed['answer'])),
        'support_score': None,
        'candidate20_ids': [],
        'final_context_ids': [],
        'latency_sec': time.perf_counter() - started,
    }


def flatten_full_rag_result(result: Dict[str,Any]) -> Dict[str,Any]:
    retr = result.get('retrieval', {})
    return {
        'answer': result.get('answer',''),
        'raw_output': result.get('raw_output',''),
        'source_ids': result.get('source_ids',[]),
        'format_ok': result.get('format_ok',False),
        'status': result.get('status',''),
        'invalid_source_ids': result.get('invalid_source_ids',[]),
        'unsupported_years': result.get('unsupported_years',[]),
        'support_score': result.get('support_score'),
        'candidate20_ids': [str(c['chunk_id']) for c in retr.get('candidates20',[])],
        'final_context_ids': [str(c['chunk_id']) for c in retr.get('final_context',[])],
        'latency_sec': result.get('latency_sec'),
    }


def evaluate_variant(variant: str, model=None, tok=None):
    done = load_benchmark_done()
    buffer = []
    pending = [r for r in benchmark_rows if (variant, r['id']) not in done]
    print(f'\n[{variant}] pending {len(pending)}/{len(benchmark_rows)}')

    for i, row in enumerate(tqdm(pending, desc=variant), 1):
        if variant in {'vanilla', 'stage1'}:
            out = run_plain_variant(model, tok, row, variant)
        elif variant == 'stage12_weights':
            out = run_stage12_weights_only(model, tok, row)
        elif variant == 'stage12_full_rag':
            out = flatten_full_rag_result(rag_chain.invoke(row['question']))
        else:
            raise ValueError(variant)

        rec = {
            'variant': variant,
            'id': row['id'],
            'type': row['type'],
            'question': row['question'],
            'reference_answer': row.get('reference_answer',''),
            'gold_source_ids': row.get('gold_source_ids_present', row.get('gold_source_ids',[])),
            'is_off_topic': bool(row.get('is_off_topic',False)),
            'expected_refusal': bool(row.get('expected_refusal',False)),
            **out,
        }
        buffer.append(rec)
        if len(buffer) >= BENCHMARK_SAVE_EVERY:
            append_jsonl(BENCHMARK_RESULTS_PATH, buffer)
            buffer = []
    append_jsonl(BENCHMARK_RESULTS_PATH, buffer)


# ---- Vanilla ----
if any(('vanilla', r['id']) not in load_benchmark_done() for r in benchmark_rows):
    print('\nLoading VANILLA...')
    m, t = load_model_variant('vanilla')
    evaluate_variant('vanilla', m, t)
    del m, t
    cleanup_cuda()
else:
    print('Vanilla already complete.')

# ---- Stage1 ----
if any(('stage1', r['id']) not in load_benchmark_done() for r in benchmark_rows):
    print('\nLoading STAGE1 merged...')
    m, t = load_model_variant('stage1')
    evaluate_variant('stage1', m, t)
    del m, t
    cleanup_cuda()
else:
    print('Stage1 already complete.')

# ---- Stage12 weights-only ----
# Nếu global final model còn sẵn, tái sử dụng để khỏi merge lần nữa.
if any(('stage12_weights', r['id']) not in load_benchmark_done() for r in benchmark_rows):
    evaluate_variant('stage12_weights', generation_model, generation_tokenizer)
else:
    print('Stage12 weights already complete.')

# ---- Full system ----
if any(('stage12_full_rag', r['id']) not in load_benchmark_done() for r in benchmark_rows):
    evaluate_variant('stage12_full_rag')
else:
    print('Full RAG already complete.')

print('Benchmark generations complete:', BENCHMARK_RESULTS_PATH)

In [ ]:
# Cell 17 — Compute final metrics + comparison table

results = read_jsonl(BENCHMARK_RESULTS_PATH)
res_df = pd.DataFrame(results)

# Keep latest record for each variant/id if notebook resumed and appended duplicates.
res_df = res_df.drop_duplicates(subset=['variant','id'], keep='last').reset_index(drop=True)

res_df['refusal'] = res_df['answer'].apply(is_refusal)
res_df['behavior_correct'] = res_df['refusal'] == res_df['expected_refusal']
res_df['token_f1'] = res_df.apply(lambda r: token_f1(r['answer'], r['reference_answer']) if not r['is_off_topic'] else np.nan, axis=1)
res_df['rougeL_f1'] = res_df.apply(lambda r: rouge_l_f1(r['answer'], r['reference_answer']) if not r['is_off_topic'] else np.nan, axis=1)
res_df['year_f1'] = res_df.apply(lambda r: year_f1(r['answer'], r['reference_answer']) if not r['is_off_topic'] else np.nan, axis=1)

# Semantic similarity in batch for historical rows with reference answer.
mask = (~res_df['is_off_topic']) & res_df['reference_answer'].astype(bool)
sem_values = [np.nan] * len(res_df)
idxs = res_df.index[mask].tolist()
if idxs:
    sims = semantic_similarity_batch(
        res_df.loc[idxs, 'answer'].tolist(),
        res_df.loc[idxs, 'reference_answer'].tolist(),
    )
    for idx, sim in zip(idxs, sims):
        sem_values[idx] = sim
res_df['semantic_similarity'] = sem_values

# Citation/retrieval metrics.
sp, sr, sf = [], [], []
r20, rf, mrr = [], [], []
for _, r in res_df.iterrows():
    p, rec, f1 = source_prf(r.get('source_ids',[]) or [], r.get('gold_source_ids',[]) or [])
    sp.append(p); sr.append(rec); sf.append(f1)
    rm = retrieval_metrics(r.get('candidate20_ids',[]) or [], r.get('final_context_ids',[]) or [], r.get('gold_source_ids',[]) or [])
    r20.append(rm['retrieval_recall20']); rf.append(rm['retrieval_recall_final']); mrr.append(rm['mrr20'])
res_df['source_precision'] = sp
res_df['source_recall'] = sr
res_df['source_f1'] = sf
res_df['retrieval_recall20'] = r20
res_df['retrieval_recall_final'] = rf
res_df['mrr20'] = mrr
res_df['source_validity'] = res_df['invalid_source_ids'].apply(lambda xs: 1.0 if not (xs or []) else 0.0)
res_df['unsupported_year_free'] = res_df['unsupported_years'].apply(lambda xs: 1.0 if not (xs or []) else 0.0)

# Group summary.
def nanmean(s):
    x = pd.to_numeric(s, errors='coerce')
    return float(x.mean()) if x.notna().any() else np.nan

summary_rows = []
for variant, g in res_df.groupby('variant', sort=False):
    hist = g[~g['is_off_topic']]
    ood = g[g['is_off_topic']]
    summary_rows.append({
        'variant': variant,
        'n': len(g),
        'semantic_similarity': nanmean(hist['semantic_similarity']),
        'token_f1': nanmean(hist['token_f1']),
        'rougeL_f1': nanmean(hist['rougeL_f1']),
        'year_f1': nanmean(hist['year_f1']),
        'behavior_accuracy_all': nanmean(g['behavior_correct']),
        'off_topic_refusal_accuracy': nanmean(ood['behavior_correct']) if len(ood) else np.nan,
        'history_behavior_accuracy': nanmean(hist['behavior_correct']) if len(hist) else np.nan,
        'format_rate': nanmean(g['format_ok']),
        # Source F1 chỉ có ý nghĩa trên câu lịch sử có gold evidence.
        'source_f1': nanmean(g[g['gold_source_ids'].apply(bool)]['source_f1']) if g['gold_source_ids'].apply(bool).any() else np.nan,
        'source_validity': nanmean(g['source_validity']) if variant in {'stage12_weights','stage12_full_rag'} else np.nan,
        'unsupported_year_free': nanmean(g['unsupported_year_free']) if variant in {'stage12_weights','stage12_full_rag'} else np.nan,
        # Retrieval metrics chỉ có ý nghĩa cho full RAG.
        'retrieval_recall20': nanmean(g['retrieval_recall20']) if variant == 'stage12_full_rag' else np.nan,
        'retrieval_recall_final': nanmean(g['retrieval_recall_final']) if variant == 'stage12_full_rag' else np.nan,
        'mrr20': nanmean(g['mrr20']) if variant == 'stage12_full_rag' else np.nan,
        'grounding_support': nanmean(g['support_score']) if variant == 'stage12_full_rag' else np.nan,
        'avg_latency_sec': nanmean(g['latency_sec']),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(BENCHMARK_SUMMARY_PATH, index=False, encoding='utf-8-sig')

print('FINAL COMPARISON')
display(summary_df.round(4))
print('Saved:', BENCHMARK_SUMMARY_PATH)

# Useful diagnostic: full RAG failures.
full_bad = res_df[(res_df['variant']=='stage12_full_rag') & (~res_df['behavior_correct'])]
print('Full RAG behavior errors:', len(full_bad))
display(full_bad[['id','type','question','answer','status','gold_source_ids','source_ids']].head(20))

## Cách đọc benchmark

Bốn cấu hình được so sánh:

1. **`vanilla`** — Qwen2.5-3B-Instruct nguyên bản, hỏi trực tiếp.
2. **`stage1`** — vanilla + Phase1 LoRA đã merge, hỏi trực tiếp theo prompt Phase1.
3. **`stage12_weights`** — Phase1 + Phase2 đã merge nhưng **không cung cấp retrieval context**. Mục đích là xem Phase2 có học hành vi abstain/grounding hay không; không kỳ vọng cấu hình này có answer-quality cao.
4. **`stage12_full_rag`** — model merge cả hai stage + FAISS + BM25 + RRF + reranker + metadata boost + OOD/source/year guards + LangChain.

Metrics chính:

- `semantic_similarity`, `token_f1`, `rougeL_f1`: độ gần đáp án gold.
- `year_f1`: độ đúng các mốc năm so với gold.
- `behavior_accuracy_all`: trả lời khi nên trả lời và từ chối khi nên từ chối.
- `off_topic_refusal_accuracy`: riêng 10 câu OOD.
- `source_f1`: source IDs model chọn so với gold evidence.
- `source_validity`: model có bịa source ID ngoài context hay không.
- `unsupported_year_free`: không sinh mốc năm ngoài evidence.
- `retrieval_recall20`: gold evidence có nằm trong 20 candidates sau RRF hay không.
- `retrieval_recall_final`: gold evidence có sống sót tới context cuối sau reranker + metadata không.
- `mrr20`: gold evidence đứng cao đến đâu trong 20 candidates.
- `grounding_support`: semantic support proxy giữa answer và chunk được cite.

Không nên chỉ nhìn một metric. Với hệ thống chống bịa, **behavior accuracy + source validity + retrieval recall + answer similarity** quan trọng hơn một điểm tổng hợp duy nhất.